# Hidden Value: Weekly Rankings

**Purpose:** season-to-date rankings from the drive table, in four parts:
1. **Team ranking**: offense and defense together
2. **Team offense ranking**
3. **Team defense ranking**
4. **Player position rankings**: QB, RB, WR, TE

**Rules this notebook follows (Design Doc v1.0)**
- **Research frame.** Garbage-time drives (Sec 11) and clock-expiration drives (Sec 14) are left out. This is different from the projection notebooks, which keep garbage time.
- **Gate (Sec 15).** Until every team has 40 non-garbage drives on offense and on defense, these are **leaders with a small-sample warning, not official rankings**. Teams under 40 are marked `*`. The status line at the top of each table says which state you are in.
- **No Drive Quality Score yet.** Its weights are not set (Sec 20), so nothing here is a DQS. Offense and defense are ranked by points per drive, the project's primary outcome (Sec 10), and every other dimension gets its own rank next to it (Sec 17).
- **No opponent adjustment.** Deferred by design (Sec 19).
- **Players.** Separate QB, RB, WR and TE boards, built like `RESEARCH_Top_Performers_v2`: stats pulled toward league average before scoring, a 0-100 SCORE, a gate that rises with the season, and a `*` with a note on anyone hurt, benched or on a small sample. The QB board includes every team's QB. Details in Section 8.

**When to run:** Tuesday morning, after Monday night's play-by-play is posted. Re-run Phase 2 (WEEKLY_1) first. Section 1 warns if the drive table or the week is incomplete.

**Long tables in VS Code:** VS Code cuts any output longer than 30 lines and shows `...` in the middle. Turn on the setting `notebook.output.scrolling` (Settings, search "notebook output scrolling") so every table scrolls instead. The Excel file always has the full tables.

**Output:** one Excel file per week, `hidden_value_rankings_through_weekNN.xlsx`, with a tab for each table. Rankings are descriptive, so re-running overwrites that week's file.

## 1. Setup and load

In [46]:
import nflreadpy as nfl
import pandas as pd, numpy as np
import os

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 240)

SEASON = 2026
FOLDER = r"C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football"
CSV = os.path.join(FOLDER, f"hidden_value_{SEASON}_drives.csv")

def to_pd(x, cols=None):
    """nflreadpy returns Polars; convert to pandas. cols keeps memory down on big pulls."""
    if cols is not None and hasattr(x, "select"):
        x = x.select([c for c in cols if c in x.columns])
    try:
        return x.to_pandas()
    except AttributeError:
        return x if cols is None else x[[c for c in cols if c in x.columns]]

drives = pd.read_csv(CSV)

PBP_COLS = ["game_id", "season_type", "week"]   # only used to check the drive table is current
pbp = to_pd(nfl.load_pbp(seasons=[SEASON]), PBP_COLS)
pbp = pbp[pbp["season_type"] == "REG"].copy()
sched = to_pd(nfl.load_schedules(seasons=[SEASON]), ["game_id", "game_type", "week", "away_team", "home_team", "home_score"])
sched = sched[sched["game_type"] == "REG"].copy()

THROUGH = int(drives["week"].max())
print(f"drive table: {len(drives)} drives, weeks 1-{THROUGH}")

# 1. The drive table must have every game in the play-by-play.
missing = sorted(set(pbp["game_id"]) - set(drives["game_id"]))
if missing:
    print(">> STALE CSV: re-run Phase 2 (WEEKLY_1) first. Games missing from the drive table:", missing)
# 2. Every game in the weeks covered should be final. A Monday night game posts overnight.
unfinished = sched[(sched["week"] <= THROUGH) & sched["home_score"].isna()]
if len(unfinished):
    print(">> NOT FINAL YET:", unfinished["game_id"].tolist(), "Those teams are missing a game. Re-run Tuesday morning.")
if not missing and not len(unfinished):
    print(f"drive table is complete through week {THROUGH}")

drive table: 2559 drives, weeks 1-8
drive table is complete through week 8


## 2. Settings (fixed, do not tune to the results)

`GATE` comes from the design doc. If it changes, log it with a date, like any other definition change.

In [47]:
GATE = 40                  # Sec 15: non-garbage drives each team needs, on offense and on defense

# player settings (gate, K values, weights) live in Section 8, carried over from RESEARCH_Top_Performers_v2

## 3. Gate check

Counted the same way as the Phase 2 table: non-garbage drives, clock drives included. Defense gets the same check, on drives faced.

In [48]:
nong = drives[drives["garbage_time"] == 0]
gate = pd.DataFrame({"off_drives": nong.groupby("posteam").size(),
                     "def_drives": nong.groupby("defteam").size()}).fillna(0).astype(int)
short = gate[(gate["off_drives"] < GATE) | (gate["def_drives"] < GATE)]
PRE_GATE = len(short) > 0

if PRE_GATE:
    STATUS = (f"THROUGH WEEK {THROUGH}: PRE-GATE LEADERS, NOT OFFICIAL RANKINGS. "
              f"{len(short)} team(s) under {GATE} drives (marked *): {', '.join(short.index)}")
else:
    STATUS = f"THROUGH WEEK {THROUGH}: OFFICIAL RANKINGS. Every team has {GATE}+ drives on offense and defense."
print(STATUS)
if PRE_GATE:
    print("\nteams under the gate:")
    print(short.sort_values("off_drives").to_string())

THROUGH WEEK 8: OFFICIAL RANKINGS. Every team has 40+ drives on offense and defense.


## 4. Team metrics

One function builds every metric for a unit. For offense it groups by the team with the ball; for defense it groups by the team on defense, so every defensive number is what that defense **allowed**.

| Metric | Definition | Design doc |
|---|---|---|
| pts_per_drive | offensive points (7/3/0) per drive | Sec 10 |
| epa_per_play, success_rate | over qualifying plays | Sec 5 |
| explosive_rate | run 10+, pass 15+, per qualifying play | Sec 6 |
| third_down_rate | conversions / attempts | Sec 5 |
| scoring_opp_rate | drives reaching the opponent 40 | Sec 7 |
| pts_per_scoring_opp | points on those drives / those drives | Sec 7 |
| red_zone_td_rate | TD drives / drives reaching the opponent 20 | Sec 7 |
| negative_play_rate | sacks, runs for loss, offensive-penalty no-plays, per qualifying play | Sec 5, 9 |
| sack_rate | sacks / dropbacks | Sec 9 |
| turnover_rate | drives with a turnover / drives | Sec 9 |
| start_own_yd | average drive start, in yards from the team's own goal line (higher is better for an offense) | Sec 8 |

In [49]:
R = drives[(drives["garbage_time"] == 0) & (drives["clock_expiration"] == 0)].copy()
R["pts_on_opp"] = R["offensive_points"] * R["reached_40"]
R["td_in_rz"] = R["td_drive"] * R["reached_20"]
R["to_drive"] = (R["turnovers"] > 0).astype(int)
R["start_own_yd"] = 100 - R["start_yardline_100"]

# True = a higher number is better for an OFFENSE. Defense uses the opposite.
METRICS = {"pts_per_drive": True, "epa_per_play": True, "success_rate": True, "explosive_rate": True,
           "third_down_rate": True, "scoring_opp_rate": True, "pts_per_scoring_opp": True,
           "red_zone_td_rate": True, "negative_play_rate": False, "sack_rate": False,
           "turnover_rate": False, "start_own_yd": True}

def unit_metrics(df, key):
    g = df.groupby(key)
    s = g[["offensive_points", "epa_sum", "qualifying_plays", "success_plays", "explosive_plays",
           "third_down_conversions", "third_down_attempts", "reached_40", "pts_on_opp", "reached_20",
           "td_in_rz", "negative_plays", "sacks", "pass_plays", "to_drive"]].sum()
    n = g.size()
    t = pd.DataFrame(index=n.index)
    t.index.name = "team"
    t["drives"] = n
    t["pts_per_drive"] = s["offensive_points"] / n
    t["epa_per_play"] = s["epa_sum"] / s["qualifying_plays"]
    t["success_rate"] = s["success_plays"] / s["qualifying_plays"]
    t["explosive_rate"] = s["explosive_plays"] / s["qualifying_plays"]
    t["third_down_rate"] = s["third_down_conversions"] / s["third_down_attempts"]
    t["scoring_opp_rate"] = s["reached_40"] / n
    t["pts_per_scoring_opp"] = s["pts_on_opp"] / s["reached_40"]
    t["red_zone_td_rate"] = s["td_in_rz"] / s["reached_20"]
    t["negative_play_rate"] = s["negative_plays"] / s["qualifying_plays"]
    t["sack_rate"] = s["sacks"] / s["pass_plays"]
    t["turnover_rate"] = s["to_drive"] / n
    t["start_own_yd"] = g["start_own_yd"].mean()
    return t

def add_ranks(t, higher_is_better):
    """Rank 1 = best. Ties share the better rank."""
    for c, hi in higher_is_better.items():
        t[c + "_rk"] = t[c].rank(ascending=not hi, method="min").astype(int)
    return t

OFF = add_ranks(unit_metrics(R, "posteam"), METRICS)
DEF = add_ranks(unit_metrics(R, "defteam"), {k: not v for k, v in METRICS.items()})
OFF["flag"] = np.where(gate["off_drives"].reindex(OFF.index) < GATE, "*", "")
DEF["flag"] = np.where(gate["def_drives"].reindex(DEF.index) < GATE, "*", "")

LEAGUE = unit_metrics(R.assign(all="NFL"), "all").iloc[0]
print("league averages (research frame):")
print(LEAGUE.drop("drives").round(3).to_string())

league averages (research frame):
pts_per_drive           2.240
epa_per_play            0.010
success_rate            0.438
explosive_rate          0.129
third_down_rate         0.400
scoring_opp_rate        0.532
pts_per_scoring_opp     4.197
red_zone_td_rate        0.646
negative_play_rate      0.127
sack_rate               0.063
turnover_rate           0.123
start_own_yd           30.668


## 5. Team offense ranking

Ordered by points per drive. The `_rk` columns rank every other dimension on its own, so you can see **how** each offense gets its points (Sec 17: most efficient, most explosive, best finishing).

`drives` here leaves out clock-expiration drives, so it can run a few lower than the gate count in Section 3. The `*` comes from the gate count.

In [50]:
SHOW = ["pts_per_drive", "epa_per_play", "success_rate", "explosive_rate", "third_down_rate",
        "scoring_opp_rate", "red_zone_td_rate", "turnover_rate", "start_own_yd"]
RANKS = [c + "_rk" for c in METRICS]

def ranked_view(t):
    v = t.sort_values(["pts_per_drive_rk", "epa_per_play_rk"]).copy()
    v.insert(0, "rank", v["pts_per_drive_rk"].astype(str) + v["flag"])
    return v

off_view = ranked_view(OFF)
print(STATUS, "\n")
print("TEAM OFFENSE, values:")
print(off_view[["rank", "drives"] + SHOW].round(3).to_string())
print("\nTEAM OFFENSE, rank in each dimension (1 = best):")
print(off_view[["rank"] + RANKS].to_string())

THROUGH WEEK 8: OFFICIAL RANKINGS. Every team has 40+ drives on offense and defense. 

TEAM OFFENSE, values:
     rank  drives  pts_per_drive  epa_per_play  success_rate  explosive_rate  third_down_rate  scoring_opp_rate  red_zone_td_rate  turnover_rate  start_own_yd
team                                                                                                                                                          
BUF     1      63          3.333         0.172         0.485           0.196            0.439             0.698             0.750          0.175        32.651
CIN     2      52          3.038         0.152         0.520           0.156            0.585             0.615             0.741          0.154        34.058
SF      3      51          2.980         0.184         0.548           0.163            0.517             0.627             0.909          0.157        30.157
DAL     4      59          2.932         0.108         0.477           0.120            0.453   

## 6. Team defense ranking

Same metrics, what each defense **allowed**. Rank 1 = best defense, so low points per drive allowed ranks first. For `start_own_yd`, a good defense keeps opponents starting deep in their own end (lower), but most of that is the defense's own offense and special teams, so read it as context.

In [51]:
def_view = ranked_view(DEF)
print(STATUS, "\n")
print("TEAM DEFENSE, values allowed:")
print(def_view[["rank", "drives"] + SHOW].round(3).to_string())
print("\nTEAM DEFENSE, rank in each dimension (1 = best):")
print(def_view[["rank"] + RANKS].to_string())

THROUGH WEEK 8: OFFICIAL RANKINGS. Every team has 40+ drives on offense and defense. 

TEAM DEFENSE, values allowed:
     rank  drives  pts_per_drive  epa_per_play  success_rate  explosive_rate  third_down_rate  scoring_opp_rate  red_zone_td_rate  turnover_rate  start_own_yd
team                                                                                                                                                          
MIN     1      67          1.328        -0.143         0.369           0.119            0.294             0.418             0.400          0.119        29.582
JAX     2      54          1.556        -0.138         0.399           0.117            0.355             0.481             0.450          0.185        28.870
SEA     3      73          1.616        -0.105         0.384           0.077            0.284             0.384             0.812          0.123        30.630
TB      4      67          1.627        -0.162         0.388           0.109            

## 7. Team ranking (offense and defense together)

Ranked by **points per drive margin**: offensive points per drive minus points per drive allowed. It uses the project's primary outcome, needs no weights, and every input is visible. EPA per play margin is shown next to it as a check, since EPA is steadier week to week. When the two ranks disagree by a lot, the team's scoring is running ahead of or behind its play-by-play, which is the over/underperformance question in Sec 18.

This is **not** the Drive Quality Score.

In [52]:
TEAM = pd.DataFrame({"off_ppd": OFF["pts_per_drive"], "def_ppd_allowed": DEF["pts_per_drive"],
                     "off_epa": OFF["epa_per_play"], "def_epa_allowed": DEF["epa_per_play"]})
TEAM["ppd_margin"] = TEAM["off_ppd"] - TEAM["def_ppd_allowed"]
TEAM["epa_margin"] = TEAM["off_epa"] - TEAM["def_epa_allowed"]
TEAM["off_rk"] = OFF["pts_per_drive_rk"]
TEAM["def_rk"] = DEF["pts_per_drive_rk"]
TEAM["ppd_margin_rk"] = TEAM["ppd_margin"].rank(ascending=False, method="min").astype(int)
TEAM["epa_margin_rk"] = TEAM["epa_margin"].rank(ascending=False, method="min").astype(int)
TEAM["rank_gap"] = TEAM["epa_margin_rk"] - TEAM["ppd_margin_rk"]   # + = scoring better than its EPA suggests
TEAM["flag"] = np.where((OFF["flag"] == "*") | (DEF["flag"] == "*"), "*", "")
TEAM = TEAM.sort_values(["ppd_margin_rk", "epa_margin_rk"])
TEAM.insert(0, "rank", TEAM["ppd_margin_rk"].astype(str) + TEAM["flag"])

print(STATUS, "\n")
print("TEAM RANKING (points per drive margin):")
print(TEAM[["rank", "ppd_margin", "off_ppd", "def_ppd_allowed", "off_rk", "def_rk",
            "epa_margin", "epa_margin_rk", "rank_gap"]].round(3).to_string())
print("\nrank_gap: + means the team ranks higher on points than on EPA (scoring ahead of its play), - means behind.")

THROUGH WEEK 8: OFFICIAL RANKINGS. Every team has 40+ drives on offense and defense. 

TEAM RANKING (points per drive margin):
     rank  ppd_margin  off_ppd  def_ppd_allowed  off_rk  def_rk  epa_margin  epa_margin_rk  rank_gap
team                                                                                                
JAX     1       1.093    2.648            1.556       8       2       0.281              1         0
SF      2       1.020    2.980            1.960       3       9       0.249              2         0
CHI     3       0.892    2.790            1.898       5       7       0.179              3         0
BUF     4       0.874    3.333            2.459       1      22       0.110              7         3
CIN     5       0.788    3.038            2.250       2      17       0.134              6         1
KC      6       0.673    2.709            2.036       7      12       0.169              4        -2
LA      7       0.536    2.507            1.971      11      10  

## 8. Player position rankings

Built the same way as `RESEARCH_Top_Performers_v2`, with three changes: a **TE** board, **every team's QB** on the QB board, and a `*` on anyone who is hurt or out of the lineup.

**How each board works**
- Season-to-date nflverse player stats plus Next Gen Stats, through the same week as the team tables.
- Every rate stat is pulled toward league average by `K` plays before it is scored. `K` is the sample size where the stat is half signal, half noise, estimated on 2021-2025 in v2. TE borrows the WR values (not estimated separately).
- `SCORE` is 0 to 100 against that position's **qualified** pool: 50 = average, 15 points = one standard deviation.
- **Gate** rises with the season: 14 attempts, 6.25 carries, 3 targets (WR) or 2 targets (TE) per team game.
- **`*` and `note`:** `*` means something about his number needs a second look. `note` says what: injured (Out, Doubtful, reserve list or inactive), missed his team's last game, lost the starting job, or a small sample.

**QB board:** every qualified QB, plus each team's current starter and each team's leader in dropbacks, even if they are under the gate. So all 32 teams are on it. A starter under the gate is scored, but his numbers are pulled hard toward average, so read his rank as a placeholder.

**RB / WR / TE boards:** qualified players are ranked. Players with at least half the gate are listed below the board so nobody disappears.

One difference from the team tables: these player stats **include garbage time** (nflverse's weekly stats are not split by game state). Weights are the v2 weights.

In [53]:
pstats = to_pd(nfl.load_player_stats(seasons=[SEASON]))
pstats = pstats[(pstats["season_type"] == "REG") & (pstats["week"] <= THROUGH)].copy()

def ngs_season(stat_type):
    """Season-to-date NGS rows (week 0). Asking for one season only keeps memory down."""
    try:
        df = to_pd(nfl.load_nextgen_stats(seasons=[SEASON], stat_type=stat_type))
    except TypeError:
        df = to_pd(nfl.load_nextgen_stats(stat_type=stat_type))
    df = df[(df["season"] == SEASON) & (df["season_type"] == "REG") & (df["week"] == 0)]
    return df.drop_duplicates("player_gsis_id").set_index("player_gsis_id")

ngs_pass, ngs_rush, ngs_rec = ngs_season("passing"), ngs_season("rushing"), ngs_season("receiving")

try:
    inj = to_pd(nfl.load_injuries(seasons=[SEASON]))
    inj = inj[inj["week"] <= THROUGH + 1]
except Exception as e:
    inj = None; print("injury reports not available:", e)
try:
    rosters_w = to_pd(nfl.load_rosters_weekly(seasons=[SEASON]), ["season", "week", "team", "gsis_id", "status"])
    rosters_w = rosters_w[rosters_w["week"] <= THROUGH + 1]
except Exception as e:
    rosters_w = None; print("weekly rosters not available:", e)

team_games = pstats.groupby("team")["week"].nunique()
team_last = pstats.groupby("team")["week"].max()
print("player stats weeks:", sorted(pstats["week"].unique().tolist()), "| NGS rows:", len(ngs_pass), len(ngs_rush), len(ngs_rec))
if inj is not None and len(inj):
    print("latest injury report: week", int(inj["week"].max()),
          "(the next week's report starts Wednesday; until then this is last week's)")

player stats weeks: [1, 2, 3, 4, 5, 6, 7, 8] | NGS rows: 35 45 115
latest injury report: week 9 (the next week's report starts Wednesday; until then this is last week's)


In [54]:
K = {
    # QB
    "epa_per_dropback": 250, "yards_per_att": 270, "int_rate": 2700, "sack_rate": 210, "cpoe": 350,
    # RB
    "rush_epa_per_carry": 350, "yards_per_carry": 230, "ryoe_per_att": 370,
    # WR and TE (TE borrows the WR values)
    "epa_per_target": 250, "yards_per_target": 140, "catch_rate": 90, "avg_separation": 35, "yac_over_expected": 55,
}
PLAYER_GATE = {"QB": 14, "RB": 6.25, "WR": 3, "TE": 2}   # attempts / carries / targets per team game

def shrink(t, name, num, den):
    """Raw rate, plus the rate pulled toward the league rate by K[name] plays."""
    lg = t[num].sum() / t[den].sum()
    t[name] = t[num] / t[den].replace(0, np.nan)
    t[name + "_adj"] = (t[num] + K[name] * lg) / (t[den] + K[name])

def shrink_ngs(t, name, ngs, value_col, count_col):
    """Same idea for a Next Gen Stats average. A player with no NGS row gets the league value."""
    v = pd.to_numeric(t["player_id"].map(ngs[value_col]), errors="coerce")
    n = pd.to_numeric(t["player_id"].map(ngs[count_col]), errors="coerce").where(v.notna(), 0).fillna(0)
    lg = (v.fillna(0) * n).sum() / n.sum()
    t[name] = v
    t[name + "_adj"] = (v.fillna(0) * n + K[name] * lg) / (n + K[name])

def z_to_100(s, ref, higher_better=True):
    """50 = average of the qualified pool (ref), 15 points per standard deviation."""
    s = pd.to_numeric(s, errors="coerce")
    if not higher_better:
        s = -s
    mu, sd = s[ref].mean(), s[ref].std(ddof=0)
    if not sd or np.isnan(sd):
        return pd.Series(50.0, index=s.index)
    return (50 + 15 * (s - mu) / sd).clip(0, 100)

def build_score(df, metrics, ref):
    tot = sum(w for _, w, _ in metrics)
    out = pd.Series(0.0, index=df.index)
    for c, w, h in metrics:
        out = out + z_to_100(df[c], ref, h) * w
    return out / tot

def aggregate(rows, sums, means=()):
    """One row per player (by ID, so a traded player is one row). Team = his most recent team."""
    rows = rows.sort_values("week")
    g = rows.groupby("player_id")
    t = g[list(sums)].sum()
    for c in means:
        t[c] = g[c].mean()
    t["player"] = g["player_display_name"].last()
    t["team"] = g["team"].last()
    t["games"] = g["week"].nunique()
    t["last_week"] = g["week"].max()
    t["team_games"] = t["team"].map(team_games)
    t["weeks_since_played"] = t["team"].map(team_last) - t["last_week"]
    return t.reset_index()

def add_status(t):
    """injured = Out or Doubtful on the latest report, or on a reserve list / inactive."""
    t["injury_report"] = ""
    if inj is not None and len(inj):
        cur = (inj[inj["week"] == inj["week"].max()].dropna(subset=["gsis_id"])
               .drop_duplicates("gsis_id").set_index("gsis_id"))
        t["injury_report"] = t["player_id"].map(cur["report_status"].fillna("listed, no status")).fillna("")
    t["roster_status"] = ""
    if rosters_w is not None and len(rosters_w):
        cur = (rosters_w.dropna(subset=["gsis_id"]).sort_values("week")
               .drop_duplicates("gsis_id", keep="last").set_index("gsis_id"))
        t["roster_status"] = t["player_id"].map(cur["status"]).fillna("")
    hurt = t["injury_report"].isin(["Out", "Doubtful"]) | t["roster_status"].isin(["RES", "INA"])
    t["status"] = np.where(hurt, "injured", "active")
    return t

def add_note(t, unit):
    """Plain-English reason for the *, empty if nothing to flag."""
    notes = []
    for r in t.itertuples():
        n = []
        if r.status == "injured":
            why = r.injury_report if r.injury_report in ("Out", "Doubtful") else {"RES": "reserve list", "INA": "inactive"}.get(r.roster_status, "")
            n.append(f"injured ({why})" if why else "injured")
        elif r.status == "bench":
            n.append("not the current starter")
        if r.weeks_since_played > 0 and r.status != "injured":
            n.append(f"missed last {int(r.weeks_since_played)} game(s)")
        if not r.qualified:
            n.append(f"small sample ({int(r.volume)} {unit})")
        notes.append("; ".join(n))
    t["note"] = notes
    t["flag"] = np.where(t["note"] != "", "*", "")
    return t

def rank_board(t, keep):
    """Rank the rows in `keep` by SCORE. Rank shows a * when the note is not empty."""
    b = t[keep].sort_values("SCORE", ascending=False).copy()
    b.insert(0, "rank", [f"{i}{f}" for i, f in zip(range(1, len(b) + 1), b["flag"])])
    return b.reset_index(drop=True)

def below_gate(t):
    return t[~t["qualified"] & (t["volume"] >= 0.5 * t["gate"])].sort_values("SCORE", ascending=False)

def show(b, cols, title, below=None, n_below=10):
    print(f"THROUGH WEEK {THROUGH}: {title}")
    print(b[cols].round(2).to_string(index=False))
    if below is not None and len(below):
        print(f"\nunder the gate (not ranked, shown so nobody disappears), top {min(n_below, len(below))} by SCORE:")
        print(below[[c for c in cols if c != "rank"]].head(n_below).round(2).to_string(index=False))

### QB

Metrics and weights from v2: EPA per dropback 30%, CPOE 25%, yards per attempt 20%, interception rate 15%, sack rate 10%.

`status`: **starting** = took the most dropbacks in his team's most recent game, **injured** = see above, **bench** = healthy and not the current starter.

In [55]:
qb = pstats[(pstats["position"] == "QB") & (pstats["attempts"] > 0)].copy()
qb["dropbacks"] = qb["attempts"] + qb["sacks_suffered"]
q = aggregate(qb, ["attempts", "dropbacks", "sacks_suffered", "passing_yards", "passing_epa", "passing_interceptions"])

shrink(q, "epa_per_dropback", "passing_epa", "dropbacks")
shrink(q, "yards_per_att", "passing_yards", "attempts")
shrink(q, "int_rate", "passing_interceptions", "attempts")
shrink(q, "sack_rate", "sacks_suffered", "dropbacks")
shrink_ngs(q, "cpoe", ngs_pass, "completion_percentage_above_expectation", "attempts")

q["volume"], q["gate"] = q["attempts"], PLAYER_GATE["QB"] * q["team_games"]
q["qualified"] = q["volume"] >= q["gate"]
q = add_status(q)

last_game = qb[qb["week"] == qb["team"].map(team_last)]
starter = last_game.sort_values("dropbacks").groupby("team")["player_id"].last()
leader = q.sort_values("dropbacks").groupby("team")["player_id"].last()
q["current_starter"] = q["player_id"] == q["team"].map(starter)
q["team_leader"] = q["player_id"] == q["team"].map(leader)
q["status"] = np.where(q["status"] == "injured", "injured", np.where(q["current_starter"], "starting", "bench"))

q["SCORE"] = build_score(q, [("epa_per_dropback_adj", 0.30, True), ("cpoe_adj", 0.25, True),
    ("yards_per_att_adj", 0.20, True), ("int_rate_adj", 0.15, False), ("sack_rate_adj", 0.10, False)], q["qualified"])
q = add_note(q, "att")

# every qualified QB, plus each team's current starter and its season leader in dropbacks
QB_BOARD = rank_board(q, q["qualified"] | q["current_starter"] | q["team_leader"])
missing_teams = sorted(set(team_games.index) - set(QB_BOARD["team"]))
QB_COLS = ["rank", "player", "team", "status", "attempts", "epa_per_dropback", "epa_per_dropback_adj",
           "yards_per_att", "cpoe", "int_rate", "sack_rate", "SCORE", "note"]
show(QB_BOARD, QB_COLS, f"QB board, {QB_BOARD['team'].nunique()} teams, {len(QB_BOARD)} QBs "
     f"(gate: {PLAYER_GATE['QB']} attempts per team game)")
if missing_teams:
    print("\n>> teams with no QB on the board (check the data):", missing_teams)

THROUGH WEEK 8: QB board, 32 teams, 35 QBs (gate: 14 attempts per team game)
rank            player team   status  attempts  epa_per_dropback  epa_per_dropback_adj  yards_per_att  cpoe  int_rate  sack_rate  SCORE                                            note
   1        Joe Burrow  CIN starting       267              0.24                  0.15           7.92  6.46      0.02       0.05  67.09                                                
   2       Brock Purdy   SF starting       211              0.29                  0.16           8.28  4.23      0.02       0.03  66.82                                                
   3       C.J. Stroud  HOU starting       247              0.12                  0.09           7.56  6.57      0.00       0.06  65.22                                                
   4        Jared Goff  DET starting       248              0.27                  0.16           7.92  1.31      0.00       0.06  64.78                                                
   

### RB

Rushing EPA per carry 30%, rush yards over expected per attempt 30%, yards per carry 20%, receiving EPA per game 20%. **Noisiest position to rate**: so much of it is the offensive line.

In [56]:
rb = pstats[pstats["position"].isin(["RB", "FB", "HB"]) & (pstats["carries"] > 0)].copy()
r = aggregate(rb, ["carries", "rushing_yards", "rushing_epa", "receiving_epa", "receptions"])

shrink(r, "rush_epa_per_carry", "rushing_epa", "carries")
shrink(r, "yards_per_carry", "rushing_yards", "carries")
shrink_ngs(r, "ryoe_per_att", ngs_rush, "rush_yards_over_expected_per_att", "rush_attempts")
r["rec_epa_per_game"] = r["receiving_epa"] / r["games"]

r["volume"], r["gate"] = r["carries"], PLAYER_GATE["RB"] * r["team_games"]
r["qualified"] = r["volume"] >= r["gate"]
r = add_status(r)
r["SCORE"] = build_score(r, [("rush_epa_per_carry_adj", 0.30, True), ("ryoe_per_att_adj", 0.30, True),
    ("yards_per_carry_adj", 0.20, True), ("rec_epa_per_game", 0.20, True)], r["qualified"])
r = add_note(r, "carries")

RB_BOARD = rank_board(r, r["qualified"])
RB_COLS = ["rank", "player", "team", "status", "carries", "yards_per_carry", "rush_epa_per_carry",
           "ryoe_per_att", "rec_epa_per_game", "SCORE", "note"]
show(RB_BOARD, RB_COLS, f"RB board, {len(RB_BOARD)} ranked (gate: {PLAYER_GATE['RB']} carries per team game)", below_gate(r))

THROUGH WEEK 8: RB board, 44 ranked (gate: 6.25 carries per team game)
rank                 player team  status  carries  yards_per_carry  rush_epa_per_carry  ryoe_per_att  rec_epa_per_game  SCORE                   note
   1          Jaylen Warren  PIT  active      119             5.92                0.13          1.93             -0.13  77.83                       
   2     Kenneth Walker III   KC  active      141             5.46                0.07          1.84             -0.30  75.18                       
   3          Chuba Hubbard  CAR  active      118             4.58                0.06          0.76              1.81  67.15                       
   4             James Cook  BUF  active      138             5.12                0.05          0.78              0.60  66.95                       
   5            Blake Corum   LA  active       75             5.19                0.14          0.49              0.50  62.37                       
   6           Jahmyr Gibbs  DET  a

### WR

EPA per target 25%, yards per target 20%, target share 15%, separation 15%, YAC over expected 15%, catch rate 10%.

In [57]:
def receiver_board(pos):
    x = pstats[(pstats["position"] == pos) & (pstats["targets"] > 0)].copy()
    t = aggregate(x, ["targets", "receptions", "receiving_yards", "receiving_epa"], means=["target_share"])
    shrink(t, "epa_per_target", "receiving_epa", "targets")
    shrink(t, "yards_per_target", "receiving_yards", "targets")
    shrink(t, "catch_rate", "receptions", "targets")
    shrink_ngs(t, "avg_separation", ngs_rec, "avg_separation", "targets")
    shrink_ngs(t, "yac_over_expected", ngs_rec, "avg_yac_above_expectation", "receptions")
    t["volume"], t["gate"] = t["targets"], PLAYER_GATE[pos] * t["team_games"]
    t["qualified"] = t["volume"] >= t["gate"]
    t = add_status(t)
    # target_share is his average share in games he played: a role stat, left unadjusted
    t["SCORE"] = build_score(t, [("epa_per_target_adj", 0.25, True), ("yards_per_target_adj", 0.20, True),
        ("target_share", 0.15, True), ("avg_separation_adj", 0.15, True), ("yac_over_expected_adj", 0.15, True),
        ("catch_rate_adj", 0.10, True)], t["qualified"])
    return add_note(t, "targets")

REC_COLS = ["rank", "player", "team", "status", "targets", "target_share", "yards_per_target",
            "epa_per_target", "catch_rate", "avg_separation", "SCORE", "note"]
w = receiver_board("WR")
WR_BOARD = rank_board(w, w["qualified"])
show(WR_BOARD, REC_COLS, f"WR board, {len(WR_BOARD)} ranked (gate: {PLAYER_GATE['WR']} targets per team game)", below_gate(w))

THROUGH WEEK 8: WR board, 86 ranked (gate: 3 targets per team game)
rank              player team  status  targets  target_share  yards_per_target  epa_per_target  catch_rate  avg_separation  SCORE                   note
   1  Jaxon Smith-Njigba  SEA  active       89          0.36             11.07            0.72        0.72            2.66  79.29                       
   2         Tee Higgins  CIN  active       61          0.23             11.90            0.85        0.70            2.37  73.33                       
   3         Zay Flowers  BAL  active       58          0.29             10.48            0.44        0.69            2.99  71.21                       
   4         Jaylin Noel  HOU  active       23          0.10             14.39            0.80        0.83            3.69  65.29                       
   5   Luther Burden III  CHI  active       56          0.22              9.77            0.31        0.75            3.39  65.19                       
   6        Dr

### TE

Same metrics and weights as WR. The gate is lower (2 targets per team game) because tight ends are targeted less. The `K` values are borrowed from WR, so treat TE scores a little more loosely.

In [58]:
te = receiver_board("TE")
TE_BOARD = rank_board(te, te["qualified"])
show(TE_BOARD, REC_COLS, f"TE board, {len(TE_BOARD)} ranked (gate: {PLAYER_GATE['TE']} targets per team game)", below_gate(te))

THROUGH WEEK 8: TE board, 45 ranked (gate: 2 targets per team game)
rank             player team  status  targets  target_share  yards_per_target  epa_per_target  catch_rate  avg_separation  SCORE                  note
   1       Travis Kelce   KC  active       42          0.19              9.76            0.52        0.69            3.94  68.66                      
   2       Hunter Henry   NE  active       34          0.15              9.59            0.55        0.85            4.35  67.11                      
   3  Harold Fannin Jr.  CLE  active       51          0.20              8.06            0.26        0.69            4.66  66.58                      
   4     Dalton Kincaid  BUF  active       35          0.18             11.46            0.50        0.77            3.16  66.03                      
   5       Brock Bowers   LV  active       61          0.29              8.97            0.54        0.67            2.39  62.94                      
   6        Sam LaPorta  D

### Who is hurt or out of the lineup

Everyone at the four positions with at least half the gate who is injured, benched, or missed his team's last game.

In [59]:
allp = pd.concat([q.assign(pos="QB"), r.assign(pos="RB"), w.assign(pos="WR"), te.assign(pos="TE")], ignore_index=True)
away = allp[(allp["volume"] >= 0.5 * allp["gate"]) &
            (allp["status"].isin(["injured", "bench"]) | (allp["weeks_since_played"] > 0))]
AWAY = away.sort_values(["pos", "SCORE"], ascending=[True, False])[
    ["pos", "player", "team", "status", "qualified", "volume", "last_week", "weeks_since_played", "SCORE",
     "injury_report", "roster_status"]]
print(AWAY.round(1).to_string(index=False))

pos                 player team  status  qualified  volume  last_week  weeks_since_played  SCORE     injury_report roster_status
 QB          Lamar Jackson  BAL   bench      False      97          8                   0   65.4 listed, no status           ACT
 QB         Marcus Mariota  WAS   bench      False      51          4                   4   52.2                             ACT
 QB           Tyson Bagent  CHI   bench      False      64          5                   3   45.3                             ACT
 QB          Tyler Huntley  BAL   bench      False     110          8                   0   45.1 listed, no status           ACT
 QB            Jalen Hurts  PHI injured       True     154          5                   3   41.0                             RES
 QB         Jameis Winston  NYG   bench       True     158          6                   1   33.5                             ACT
 QB          Jalon Daniels   TB   bench      False      98          6                   2   33.2 

## 9. Export

In [60]:
out_path = os.path.join(FOLDER, f"hidden_value_rankings_through_week{THROUGH:02d}.xlsx")
notes = pd.DataFrame({"item": ["status", "frame", "team ranking", "players", "made"],
                      "value": [STATUS,
                                "garbage-time and clock-expiration drives excluded (Design Doc Sec 11, 14)",
                                "points per drive margin; not the Drive Quality Score; no opponent adjustment",
                                "v2 method: stats pulled to average by K plays, 0-100 SCORE vs qualified pool; "
                                "gate per team game: QB 14 att, RB 6.25 carries, WR 3 / TE 2 targets; "
                                "* = injured, benched, missed last game, or small sample (see note); includes garbage time",
                                pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")]})
with pd.ExcelWriter(out_path) as xw:
    notes.to_excel(xw, sheet_name="Notes", index=False)
    TEAM.round(3).to_excel(xw, sheet_name="Team")
    off_view.round(3).to_excel(xw, sheet_name="Offense")
    def_view.round(3).to_excel(xw, sheet_name="Defense")
    QB_BOARD[QB_COLS].round(3).to_excel(xw, sheet_name="QB", index=False)
    RB_BOARD[RB_COLS].round(3).to_excel(xw, sheet_name="RB", index=False)
    WR_BOARD[REC_COLS].round(3).to_excel(xw, sheet_name="WR", index=False)
    TE_BOARD[REC_COLS].round(3).to_excel(xw, sheet_name="TE", index=False)
    AWAY.round(1).to_excel(xw, sheet_name="Hurt or out", index=False)
    gate.to_excel(xw, sheet_name="Gate")
print("saved", out_path)

saved C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football\hidden_value_rankings_through_week08.xlsx
